# 02. Feature Engineering

1. 원논문 공식 코드의 셀 정제 규칙을 적용해 수명 라벨을 정리합니다.
2. 사이클 100까지의 기록으로 셀 단위 피처를 만듭니다(`src/features.py`).
3. 배치별 상관, 다중공선성(VIF), 학습 범위 이탈을 확인하고 피처셋을 정합니다.

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.color': '#e6e6e6', 'font.size': 10, 'figure.dpi': 110})
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)
C = {'Batch 1': '#1F4E79', 'Batch 2': '#D9822B', 'Batch 3': '#2E8B74'}
B = ['Batch 1', 'Batch 2', 'Batch 3']
from features import build_features, FEATURE_SETS
from config import CONTINUED_CELLS, NOT_FINISHED_CELLS, NOISY_B3_CELLS
df = build_features()
L = df[df.in_model]

## 1. 라벨 정제 (원논문 규칙)

출처: `github.com/rdbraatz/data-driven-prediction-of-battery-cycle-life-before-capacity-degradation` 의 `Load Data.ipynb`, `LoadData.m`

| 대상 | 원논문 처리 | 이 저장소 |
|---|---|---|
| b1c0~c4 | 2017-06-30 파일에서 이어 측정한 사이클 수(662·981·1060·208·482)를 수명에 더함 | 같은 값으로 보정 |
| b1c8·10·12·13·22 | 80% 용량 미도달로 삭제 | 제외 |
| b3c2·23·32·37·42·43 | 노이즈 채널로 삭제 | 제외 |
| Batch 2 VarCharge·SLOWCYCLE 8셀 | (과제 배치라 원논문에 없음) 수명 라벨 없음 | 제외 |

보정 수명은 과제 데이터에 없는 파일에서 온 값이므로, `train.py`에서 제공 라벨 46셀과 10셀 전부 제외 36셀로 민감도를 함께 봅니다.

In [2]:
df[df.cell_uid.isin(list(CONTINUED_CELLS) + NOT_FINISHED_CELLS + NOISY_B3_CELLS) | (df.exclude_reason != '')][
    ['cell_uid', 'batch', 'policy', 'cycle_life_provided', 'cycle_life', 'end_QD', 'exclude_reason', 'label_corrected']]

,cell_uid,batch,policy,cycle_life_provided,cycle_life,end_QD,exclude_reason,label_corrected
0,b1c000,Batch 1,3.6C(80%)-3.6C,1190.0,1852.0,1.026210,,True
1,b1c001,Batch 1,3.6C(80%)-3.6C,1179.0,2160.0,1.038225,,True
2,b1c002,Batch 1,3.6C(80%)-3.6C,1177.0,2237.0,1.043279,,True
3,b1c003,Batch 1,4C(80%)-4C,1226.0,1434.0,0.970921,,True
4,b1c004,Batch 1,4C(80%)-4C,1227.0,1709.0,1.021960,,True
8,b1c008,Batch 1,5.4C(40%)-3.6C,879.0,NaN,0.969025,not_reached_eol,False
10,b1c010,Batch 1,5.4C(50%)-3C,906.0,NaN,0.961019,not_reached_eol,False
12,b1c012,Batch 1,5.4C(50%)-3.6C,902.0,NaN,0.913119,not_reached_eol,False
13,b1c013,Batch 1,5.4C(50%)-3.6C,897.0,NaN,0.923136,not_reached_eol,False
22,b1c022,Batch 1,6C(30%)-3.6C,891.0,NaN,0.963279,not_reached_eol,False


## 2. 피처 정의 (모두 사이클 2~100만 사용)

| 피처 | 정의 | 단위 | 근거 |
|---|---|---|---|
| `dq_logvar` | log10 Var_V(Q100(V) − Q10(V)), 전압 1,000점에 대한 분산 | log Ah² | Q3 세 배치 공통 최강 신호 |
| `qd_mean` | 사이클 2~100 방전 용량 평균 (0 < QD < 1.5Ah만) | Ah | Q2 용량 정보의 추가 몫 |
| `qd_slope` | 사이클 10~100 방전 용량 선형 기울기 | mAh/cycle | Q2 초기 감소 속도 |
| `ir_change` | IR(91~100 평균) − IR(2~10 평균), IR > 0만 | mΩ | Q5 저항 변화 |
| `tavg_median` | 사이클 평균 온도의 중앙값 | °C | Q5 발열, 극단값에 강한 중앙값 |
| `avg_crate_0_80`, `switch_soc` | 정책 문자열에서 계산한 0→80% 평균 C-rate, 전환 SOC | C, % | Q4 보조 실험 전용 |

참고용(선택하지 않음): `dq_logabsmin`, `dq_mean`(log 분산과 VIF 수백으로 중복), `dq_skew`, `dq_kurt`(B2·B3에서 약함), `chargetime_median`(배치마다 방향 반대)

In [3]:
cols = ['dq_logvar', 'qd_mean', 'qd_slope', 'ir_change', 'tavg_median', 'avg_crate_0_80']
L.groupby('batch')[cols].describe().T.round(3)

batch                 Batch 1  Batch 2  Batch 3
dq_logvar      count   41.000   39.000   40.000
               mean    -3.910   -3.596   -4.150
               std      0.399    0.394    0.248
               min     -5.015   -4.449   -4.695
               25%     -4.055   -3.674   -4.288
               50%     -3.838   -3.487   -4.151
               75%     -3.667   -3.336   -4.059
               max     -3.350   -3.086   -3.452
qd_mean        count   41.000   39.000   40.000
               mean     1.081    1.095    1.066
               std      0.008    0.011    0.008
               min      1.060    1.067    1.045
               25%      1.078    1.092    1.063
               50%      1.082    1.098    1.068
               75%      1.086    1.102    1.071
               max      1.097    1.111    1.077
qd_slope       count   41.000   39.000   40.000
               mean    -0.018    0.004   -0.029
               std      0.029    0.073    0.048
               min     -0.149   -0.186   -0.205
               25%     -0.021   -0.046   -0.024
               50%     -0.014    0.020   -0.017
               75%     -0.005    0.054   -0.010
               max      0.017    0.126    0.000
ir_change      count   41.000   33.000   40.000
               mean    -0.027   -0.139   -0.020
               std      0.641    0.303    0.106
               min     -3.973   -0.752   -0.219
               25%      0.038   -0.440   -0.080
               50%      0.088   -0.053   -0.033
               75%      0.128    0.112    0.005
               max      0.228    0.358    0.385
tavg_median    count   41.000   39.000   40.000
               mean    32.558   32.923   34.107
               std      0.970    1.159    1.256
               min     29.848   29.850   31.246
               25%     31.994   32.165   33.084
               50%     32.700   33.019   34.128
               75%     33.316   33.637   35.137
               max     33.873   35.675   36.068
avg_crate_0_80 count   41.000   39.000   40.000
               mean     4.453    4.801    4.801
               std      0.426    0.004    0.005
               min      3.600    4.790    4.794
               25%      4.200    4.800    4.795
               50%      4.402    4.800    4.802
               75%      4.800    4.804    4.804
               max      5.400    4.807    4.813

In [4]:
print('ir_change 결측(IR이 초기 구간에서 모두 0):', L[L.ir_change.isna()].cell_uid.tolist())
print('→ 학습 fold의 중앙값으로 대치(Pipeline 안에서 fit)')

ir_change 결측(IR이 초기 구간에서 모두 0): ['b2c041', 'b2c042', 'b2c043', 'b2c044', 'b2c045', 'b2c046']
→ 학습 fold의 중앙값으로 대치(Pipeline 안에서 fit)


## 3. 배치별 상관과 다중공선성

In [5]:
allf = ['dq_logvar', 'dq_logabsmin', 'dq_mean', 'dq_skew', 'dq_kurt', 'qd_mean', 'qd_slope', 'ir_change', 'tavg_median', 'chargetime_median', 'avg_crate_0_80']
corr = pd.DataFrame({b: [spearmanr(g[f], g.cycle_life, nan_policy='omit')[0] for f in allf] for b, g in L.groupby('batch')}, index=allf)
corr['sign_consistent'] = (np.sign(corr[B]).nunique(axis=1) == 1)
corr.round(2)

,Batch 1,Batch 2,Batch 3,sign_consistent
dq_logvar,-0.88,-0.71,-0.76,True
dq_logabsmin,-0.85,-0.66,-0.74,True
dq_mean,0.84,0.70,0.72,True
dq_skew,-0.58,-0.15,-0.14,True
dq_kurt,0.24,0.50,0.19,True
qd_mean,0.23,0.12,0.25,True
qd_slope,0.57,-0.27,0.16,False
ir_change,-0.24,0.28,-0.13,False
tavg_median,-0.39,0.17,0.02,False
chargetime_median,0.63,-0.81,0.12,False


In [6]:
def vif(X):
    X = X.dropna(); Z = (X - X.mean()) / X.std()
    return pd.Series(np.diag(np.linalg.inv(np.corrcoef(Z.values.T))), index=X.columns)
b1 = L[L.batch == 'Batch 1']
pd.DataFrame({'all candidates': vif(b1[allf]), 'F2_full': vif(b1[FEATURE_SETS['F2_full']])}).round(1)

,all candidates,F2_full
avg_crate_0_80,391.0,NaN
chargetime_median,415.9,NaN
dq_kurt,10.3,NaN
dq_logabsmin,359.7,NaN
dq_logvar,427.0,2.1
dq_mean,35.8,NaN
dq_skew,10.2,NaN
ir_change,1.5,1.3
qd_mean,1.5,1.2
qd_slope,5.5,1.6


ΔQ 요약 통계끼리, 충전시간과 평균 C-rate끼리 VIF가 수십~수백입니다. 기본 피처셋(F2)은 모두 VIF 10 미만이고, 남은 상관은 ElasticNet 규제로 다룹니다.

## 4. 학습 범위 이탈 (라벨 없이 X만으로 확인)

테스트 배치의 정답은 보지 않고, 입력 피처가 Batch 1 범위를 얼마나 벗어나는지만 확인합니다.

In [7]:
rows = []
for c in FEATURE_SETS['F2_full']:
    mu, sd, lo, hi = b1[c].mean(), b1[c].std(), b1[c].min(), b1[c].max()
    for b in ['Batch 2', 'Batch 3']:
        g = L[L.batch == b][c]
        rows.append({'feature': c, 'batch': b, 'mean_z_vs_B1': (g.mean() - mu) / sd, 'n_outside_B1_range': int(((g < lo) | (g > hi)).sum()), 'n': g.notna().sum()})
pd.DataFrame(rows).pivot(index='feature', columns='batch').round(2)

mean_z_vs_B1         n_outside_B1_range               n        
batch            Batch 2 Batch 3            Batch 2 Batch 3 Batch 2 Batch 3
feature                                                                    
dq_logvar           0.79   -0.60                 11       0      39      40
ir_change          -0.17    0.01                  3       2      33      40
qd_mean             1.81   -2.07                 20       8      39      40
qd_slope            0.75   -0.39                 21       3      39      40
tavg_median         0.38    1.60                  6      23      39      40

Batch 2는 `qd_mean`이 Batch 1보다 평균 1.8 표준편차 높고 39셀 중 20개가 범위 밖입니다. `dq_logvar`도 11개가 범위 밖입니다.
Batch 3는 `dq_logvar`가 모두 범위 안이지만, `qd_mean`은 평균 2.1 표준편차 낮고 `tavg_median`(온도)은 40셀 중 23개가 범위 밖입니다.

## 5. 피처셋

| 피처셋 | 구성 | 역할 |
|---|---|---|
| F0_variance | dq_logvar | 원논문 Variance model, 기준선 |
| F1_capacity | F0 + qd_mean, qd_slope | 용량 정보 추가 |
| F2_full | F1 + ir_change, tavg_median | 저항·온도 추가 |
| F3_full_policy | F2 + avg_crate_0_80, switch_soc | 보조 실험(선택 대상 아님) |